### Preprocessing and features engineering

Now we have a clean dataset with tennis matches starting from **2005-07-04** until **2026** and we need to preprocess it, by reducing the number of features and creating new ones to improve the model performance. Afterwards, we will split the dataset into training and test sets and start the model training.

In [295]:
import pandas as pd
import numpy as np

RAW_TRAINING_PATH = "../data/training/tennis_training.xlsx"
RAW_TESTING_PATH = "../data/testing/tennis_testing.xlsx"

raw_train_df = pd.read_excel(RAW_TRAINING_PATH)
raw_test_df = pd.read_excel(RAW_TESTING_PATH)
raw_split_index = len(raw_train_df)

# Prepare both raw splits together to keep chronological historical features consistent.
df_raw = pd.concat([raw_train_df, raw_test_df], ignore_index=True)
df_raw.head()

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,...,W3,L3,W4,L4,W5,L5,B365W,B365L,Wsets,Lsets
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,Robredo T.,Tabara M.,...,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00,2.0,0.0
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,Vinciguerra A.,Ryderstedt M.,...,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66,2.0,0.0
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,Verdasco F.,Pospisil J.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,Ginepri R.,Oudsema S.,...,6.0,0.0,NaN,NaN,NaN,NaN,1.12,5.50,2.0,1.0
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,Spadea V.,Popp A.,...,6.0,2.0,NaN,NaN,NaN,NaN,2.50,1.50,2.0,1.0


Firstly, we must randomly swap the winner and loser players in each match, to avoid the model learning about choosing always the first player as the winner, causing a data leakage.

In [296]:
df_prepared = df_raw.copy()

# Randomly assign the winner to Player 1 or Player 2.
rng = np.random.default_rng(42)
winner_is_player_1 = rng.random(len(df_prepared)) < 0.5

df_prepared["Player_1"] = np.where(
    winner_is_player_1, df_raw["Winner"], df_raw["Loser"]
)
df_prepared["Player_2"] = np.where(
    winner_is_player_1, df_raw["Loser"], df_raw["Winner"]
)

# Target: 1 if Player 1 wins, otherwise 0.
df_prepared["y"] = winner_is_player_1.astype(int)

# Align player statistics with the randomized player positions.
for name, winner_column, loser_column in [
    ("Rank", "WRank", "LRank"),
    ("Pts", "WPts", "LPts"),
    ("Odds", "B365W", "B365L"),
    ("Sets", "Wsets", "Lsets"),
    ("games1", "W1", "L1"),
    ("games2", "W2", "L2"),
    ("games3", "W3", "L3"),
    ("games4", "W4", "L4"),
    ("games5", "W5", "L5"),
]:
    df_prepared[f"{name}_1"] = np.where(
        winner_is_player_1,
        df_raw[winner_column],
        df_raw[loser_column],
    )
    df_prepared[f"{name}_2"] = np.where(
        winner_is_player_1,
        df_raw[loser_column],
        df_raw[winner_column],
    )
    
df_prepared.head()

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,...,games1_1,games1_2,games2_1,games2_2,games3_1,games3_2,games4_1,games4_2,games5_1,games5_2
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,Robredo T.,Tabara M.,...,5.0,7.0,0.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,Vinciguerra A.,Ryderstedt M.,...,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,Verdasco F.,Pospisil J.,...,2.0,6.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,Ginepri R.,Oudsema S.,...,2.0,6.0,7.0,6.0,0.0,6.0,NaN,NaN,NaN,NaN
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,Spadea V.,Popp A.,...,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN


#### Numerical features

We can start by creating simple engineered features, such as the differences between the players' ranks, points, and betting market odds.

We can also use logarithmic differences. This is useful because these variables are not linear.

**Example 1**

- Rank 1 vs 10
- Rank 101 vs 110

The absolute difference is the same (9), but the first difference is much more meaningful than the second one.

**Example 2**

- Points 9000 vs 8000
- Points 1200 vs 1000

Same considerations as example 1.

**Example 3**

- Odds 1.20 vs 1.50
- Odds 4.00 vs 4.30

Same absolute difference, but the second one is much more uncertain than the first one.

For betting odds, we use a log-ratio instead of a raw difference. This is closer to the market implied probability: positive values mean Player 1 is favored by the market, negative values mean Player 2 is favored.

We keep both raw differences and logarithmic comparisons. Logistic Regression benefits from the proportional logarithmic versions, while XGBoost can use the raw differences to learn thresholds. We do not calculate clipping limits on the complete dataset, because that would use information from the future test period.

In [297]:
p1_rank, p2_rank = df_prepared["Rank_1"], df_prepared["Rank_2"]
p1_pts, p2_pts = df_prepared["Pts_1"], df_prepared["Pts_2"]
p1_odds, p2_odds = df_prepared["Odds_1"], df_prepared["Odds_2"]

# Raw differences are retained for XGBoost.
df_prepared["Rank_Diff"] = p1_rank - p2_rank
df_prepared["Points_Diff"] = p1_pts - p2_pts

# Logarithmic comparisons express proportional advantages for Logistic Regression.
# Positive values always indicate an advantage for Player 1.
df_prepared["Rank_Log_Ratio"] = np.log(p2_rank) - np.log(p1_rank)
df_prepared["Points_Log_Ratio"] = np.log1p(p1_pts) - np.log1p(p2_pts)


# Market log-odds ratio: positive means Player 1 is favored by the market.
# This is more informative than a raw odds difference because betting odds are multiplicative.
valid_odds = (p1_odds > 0) & (p2_odds > 0)
df_prepared["Odds_Diff"] = np.nan
df_prepared.loc[valid_odds, "Odds_Diff"] = np.log(
    p2_odds[valid_odds] / p1_odds[valid_odds]
)

In [298]:
numerical_features = [
    "Rank_Diff", "Points_Diff", 
    "Rank_Log_Ratio", "Points_Log_Ratio", 
    "Odds_Diff",
]

We can convert **Best of** feature into a binary feature **Best of 5**, which is 1 if the match is best of 5 sets and 0 otherwise.

In [299]:
df_prepared["Best_of_5"] = (df_raw["Best of"] == 5).astype(int)
numerical_features += ["Best_of_5"]

We use the cleaned players dataset to create an age-difference feature.

The players file already contains match-compatible keys such as `Ruud C.` and keeps duplicated keys.
Here we only select a date of birth when exactly one candidate gives a plausible age at match date.


In [300]:
PLAYERS_PATH = "../data/training/tennis_players.xlsx"
players_df = pd.read_excel(PLAYERS_PATH)
players_df["dob"] = pd.to_datetime(players_df["dob"], errors="coerce")

MIN_PLAYER_AGE = 16
MAX_PLAYER_AGE = 45

# The player keys are already cleaned in notebook 01.
player_dob_candidates = (
    players_df
    .dropna(subset=["player_key", "dob"])
    .groupby("player_key")["dob"]
    .apply(list)
    .to_dict()
)

def get_plausible_age(player_name, match_date):
    # Return an age only when one DOB candidate is plausible for this match date.
    plausible_ages = []

    for dob in player_dob_candidates.get(player_name, []):
        age = (match_date - dob).days / 365.25
        if MIN_PLAYER_AGE <= age <= MAX_PLAYER_AGE:
            plausible_ages.append(age)

    # If multiple candidates remain, keep NaN instead of guessing.
    if len(plausible_ages) == 1:
        return plausible_ages[0]

    return np.nan

age_diff = []

for row in df_prepared.itertuples(index=False):
    age_1 = get_plausible_age(row.Player_1, row.Date)
    age_2 = get_plausible_age(row.Player_2, row.Date)
    age_diff.append(age_1 - age_2)

df_prepared["Age_Diff"] = age_diff
numerical_features += ["Age_Diff"]

print("Age_Diff coverage:", f"{df_prepared['Age_Diff'].notna().mean():.2%}")
print("Extreme age differences > 20 years:", (df_prepared["Age_Diff"].abs() > 20).sum())


Age_Diff coverage: 75.41%
Extreme age differences > 20 years: 10


### Hand matchup

In [301]:
"""Left-handed matchup, from players_df already loaded above."""
player_hand = (
    players_df
    .dropna(subset=["player_key", "hand"])
    .groupby("player_key")["hand"]
    .agg(lambda values: values.iloc[0] if values.nunique() == 1 else np.nan)
    .to_dict()
)

def is_left_handed(player_name):
    return player_hand.get(player_name) == "L"

# Check if the two players have different handedness.
df_prepared["Left_Handed_Matchup"] = (
    df_prepared["Player_1"].eq("L") ^ df_prepared["Player_2"].eq("L")
).astype(int)

numerical_features += ["Left_Handed_Matchup"]

print("Left-handed matchups:", df_prepared["Left_Handed_Matchup"].sum(), "/", len(df_prepared))

Left-handed matchups: 0 / 54938


#### Advanced features

We can improve the model performance by creating *advanced* features about players ELO ratings, fatigue and head to head statistics.

Firstly, we sort the dataset by date to avoid data leakage

In [302]:
# Elo must be calculated in chronological order.
df_prepared["Date"] = pd.to_datetime(
    df_prepared["Date"],
    errors="raise",
)

# sort dataframe by date to avoid data leakage when calculating Elo ratings
df_prepared = (
    df_prepared
    .sort_values("Date", kind="stable")
    .reset_index(drop=True) # reset index after sorting
)

#### Fatigue

In [303]:
from collections import defaultdict, deque

FATIGUE_WINDOW_DAYS = 10
FATIGUE_DECAY_DAYS = 3.0
DEFAULT_REST_DAYS = 30

recent_matches = defaultdict(list)
fatigue_diff = []


def player_fatigue(player, match_date):
    """Return recent match load before the current match."""
    fatigue = 0.0

    for previous_date in recent_matches[player]:
        days_since_match = (match_date - previous_date).days

        if 0 < days_since_match <= FATIGUE_WINDOW_DAYS:
            fatigue += np.exp(-days_since_match / FATIGUE_DECAY_DAYS)

    return fatigue


def player_rest_days(player, match_date):
    """Return days since previous match, using a neutral value for new players."""
    if not recent_matches[player]:
        return DEFAULT_REST_DAYS
    return (match_date - recent_matches[player][-1]).days


def update_fatigue(player_1, player_2, match_date):
    """Store pre-match fatigue difference, then update recent match dates."""
    fatigue_1 = player_fatigue(player_1, match_date)
    fatigue_2 = player_fatigue(player_2, match_date)
    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    # Store pre-match fatigue difference to avoid data leakage.
    fatigue_diff.append(fatigue_1 - fatigue_2)

    # Update match history only after computing the feature.
    recent_matches[player_1].append(match_date)
    recent_matches[player_2].append(match_date)

    return rest_days_1, rest_days_2

#### ELO ratings

Each player starts with an ELO rating of 1500. First, we compute the expected probability that Player 1 wins:

$$ E_1 = \frac{1}{1 + 10^{\frac{R_2 - R_1}{400}}} $$

Then, after the match, we update the rating using:

$$ \Delta R = K \cdot (S_1 - E_1) $$

where:

- $R_1, R_2$ are the pre-match ELO ratings of Player 1 and Player 2
- $E_1$ is the expected probability that Player 1 wins
- $S_1$ is the actual result: 1 if Player 1 wins, 0 otherwise
- $K$ controls how strongly ratings are updated

$K$ is dynamically reduced as a player accumulates matches, because the rating becomes more reliable:

$$ K = \max( \; \texttt{K\_MIN}, \frac{\texttt{K\_BASE}}{1 + \frac{\texttt{matches\_played}}{100}} \; ) $$

with $\texttt{K\_MIN} = 8, \; \texttt{K\_BASE} = 32$.

Following Tennis Abstract, surface ELO is used as a 50/50 blend between the player's overall ELO and raw surface ELO. The article also discusses absence handling, but the exact penalty is not fully specified, so here long absences only increase the K factor slightly instead of subtracting rating points directly.

In [304]:
K_MIN = 8.0
K_BASE = 32.0
BASE_ELO = 1500.0
SURFACE_BLEND_WEIGHT = 0.5

matches_played = {}
matches_played_surface = {}
elo_ratings = {}
elo_diff = []
surface_elo_ratings = {}
surface_elo_diff = []

ABSENCE_THRESHOLD_DAYS = 90
ABSENCE_K_MULTIPLIER = 1.25

def absence_k_multiplier(rest_days):
    return ABSENCE_K_MULTIPLIER if rest_days >= ABSENCE_THRESHOLD_DAYS else 1.0


def dynamic_k(played):
    return max(K_MIN, K_BASE / (1 + played / 100))


def blended_surface_rating(overall_rating, surface_rating):
    return (1 - SURFACE_BLEND_WEIGHT) * overall_rating + SURFACE_BLEND_WEIGHT * surface_rating


def expected_score(rating_1, rating_2):
    return 1 / (1 + 10 ** ((rating_2 - rating_1) / 400))


def update_elo_ratings(player_1, player_2, surface, y, rest_days_1, rest_days_2):
    elo_1 = elo_ratings.get(player_1, BASE_ELO)
    elo_2 = elo_ratings.get(player_2, BASE_ELO)
    surf_elo_1 = surface_elo_ratings.get((player_1, surface), BASE_ELO)
    surf_elo_2 = surface_elo_ratings.get((player_2, surface), BASE_ELO)

    blended_1 = blended_surface_rating(elo_1, surf_elo_1)
    blended_2 = blended_surface_rating(elo_2, surf_elo_2)

    expected_1 = expected_score(elo_1, elo_2)
    expected_surface_1 = expected_score(blended_1, blended_2)

    elo_diff.append(elo_1 - elo_2)
    surface_elo_diff.append(blended_1 - blended_2)

    k1 = dynamic_k(matches_played.get(player_1, 0)) * absence_k_multiplier(rest_days_1)
    k2 = dynamic_k(matches_played.get(player_2, 0)) * absence_k_multiplier(rest_days_2)
    elo_ratings[player_1] = elo_1 + k1 * (y - expected_1)
    elo_ratings[player_2] = elo_2 - k2 * (y - expected_1)

    k_surf_1 = dynamic_k(matches_played_surface.get((player_1, surface), 0)) * absence_k_multiplier(rest_days_1)
    k_surf_2 = dynamic_k(matches_played_surface.get((player_2, surface), 0)) * absence_k_multiplier(rest_days_2)
    surface_elo_ratings[(player_1, surface)] = surf_elo_1 + k_surf_1 * (y - expected_surface_1)
    surface_elo_ratings[(player_2, surface)] = surf_elo_2 - k_surf_2 * (y - expected_surface_1)

    matches_played[player_1] = matches_played.get(player_1, 0) + 1
    matches_played[player_2] = matches_played.get(player_2, 0) + 1
    matches_played_surface[(player_1, surface)] = matches_played_surface.get((player_1, surface), 0) + 1
    matches_played_surface[(player_2, surface)] = matches_played_surface.get((player_2, surface), 0) + 1

#### Recent form

We can store the last 5 matches played by each player using a queue and compute the **recent form** as the average of the last 5 matches played.

E.g. 
- $[1, 0, 1, 0, 1] \implies 0.6$

- $[1] \implies 1.0$

- $[0, 0, 1] \implies 0.33$

We can also try a weighted recent form, assigning an higher weight to most important matches as

$$ \texttt{weighted\_recent\_form} = \texttt{result} \cdot \frac{\texttt{opponent\_ELO}}{\texttt{BASE\_ELO}} $$

In [305]:
RECENT_FORM_WINDOW = 5
recent_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_form_diff = []

recent_surface_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_surface_form_diff = []

def update_recent_results(player_1, player_2, surface, y):
    form_1 = np.mean(recent_results[player_1]) if recent_results[player_1] else 0.5
    form_2 = np.mean(recent_results[player_2]) if recent_results[player_2] else 0.5
    form_surf_1 = np.mean(recent_surface_results[(player_1, surface)]) if recent_surface_results[(player_1, surface)] else 0.5
    form_surf_2 = np.mean(recent_surface_results[(player_2, surface)]) if recent_surface_results[(player_2, surface)] else 0.5

    recent_form_diff.append(form_1 - form_2)
    recent_surface_form_diff.append(form_surf_1 - form_surf_2)

    recent_results[player_1].append(y)
    recent_results[player_2].append(1 - y)
    recent_surface_results[(player_1, surface)].append(y)
    recent_surface_results[(player_2, surface)].append(1 - y)

We can also compute the **dominance form**, by considering the number of games won by each player in the last matches, instead of just the match result.

In [306]:
"""Compute number of games won by each player """
games_1 = df_prepared[["games1_1", "games2_1", "games3_1", "games4_1", "games5_1"]]
games_2 = df_prepared[["games1_2", "games2_2", "games3_2", "games4_2", "games5_2"]]

df_prepared["Games_P1"] = games_1.sum(axis=1, skipna=True)
df_prepared["Games_P2"] = games_2.sum(axis=1, skipna=True)

print(df_prepared[["Games_P1", "Games_P2", "Wsets", "Lsets", "y"]].head())

   Games_P1  Games_P2  Wsets  Lsets  y
0       5.0      13.0    2.0    0.0  0
1      12.0       4.0    2.0    0.0  1
2       6.0      12.0    2.0    0.0  0
3       9.0      18.0    2.0    1.0  0
4      16.0      11.0    2.0    1.0  1


In [307]:
DOMINANCE_FORM_WINDOW = 5
weighted_dominance_results = defaultdict(lambda: deque(maxlen=DOMINANCE_FORM_WINDOW))
dominance_form_diff = []

def update_dominance_form(player_1, player_2, games_won_1, games_total_1):

    dom_1 = np.mean(weighted_dominance_results[player_1]) if weighted_dominance_results[player_1] else 0.5
    dom_2 = np.mean(weighted_dominance_results[player_2]) if weighted_dominance_results[player_2] else 0.5

    # save pre-match, avoiding leakage
    dominance_form_diff.append(dom_1 - dom_2)

    # ratio of games won on this match
    games_ratio_1 = games_won_1 / games_total_1 if games_total_1 > 0 else 0.5
    games_ratio_2 = 1 - games_ratio_1

    weighted_dominance_results[player_1].append(games_ratio_1)
    weighted_dominance_results[player_2].append(games_ratio_2)

#### Head to head statistics

Head-to-head features count previous wins between the same two players, globally and on the current surface. Values are stored before updating the current match to avoid data leakage.

In [308]:
h2h_results = defaultdict(lambda: {"wins": 0, "losses": 0})
h2h_surface_results = defaultdict(lambda: {"wins": 0, "losses": 0})

h2h_diff = []
h2h_surface_diff = []


def update_h2h(player_1, player_2, surface, y):
    """Store pre-match head-to-head differences, then update matchup history."""
    key_1 = (player_1, player_2)
    key_2 = (player_2, player_1)

    surface_key_1 = (player_1, player_2, surface)
    surface_key_2 = (player_2, player_1, surface)

    # Store pre-match H2H difference to avoid data leakage.
    h2h_diff.append(
        h2h_results[key_1]["wins"] - h2h_results[key_1]["losses"]
    )

    h2h_surface_diff.append(
        h2h_surface_results[surface_key_1]["wins"]
        - h2h_surface_results[surface_key_1]["losses"]
    )

    # Update H2H only after computing the feature.
    if y == 1:
        h2h_results[key_1]["wins"] += 1
        h2h_results[key_2]["losses"] += 1

        h2h_surface_results[surface_key_1]["wins"] += 1
        h2h_surface_results[surface_key_2]["losses"] += 1
    else:
        h2h_results[key_1]["losses"] += 1
        h2h_results[key_2]["wins"] += 1

        h2h_surface_results[surface_key_1]["losses"] += 1
        h2h_surface_results[surface_key_2]["wins"] += 1

In [309]:
for row in df_prepared.itertuples(index=False):
    player_1, player_2 = row.Player_1, row.Player_2
    match_date, surface = row.Date, row.Surface
    y = row.y

    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    fatigue_diff.append(player_fatigue(player_1, match_date) - player_fatigue(player_2, match_date))
    update_recent_results(player_1, player_2, surface, y)
    update_elo_ratings(player_1, player_2, surface, y, rest_days_1, rest_days_2)

    games_won_P1 = row.Games_P1
    games_total = row.Games_P1 + row.Games_P2
    update_dominance_form(player_1, player_2, games_won_P1, games_total)
    update_h2h(player_1, player_2, surface, y)

    recent_matches[player_1].append(match_date)
    recent_matches[player_2].append(match_date)

df_prepared["Recent_Form_Diff"] = recent_form_diff

df_prepared["Dominance_Form_Diff"] = dominance_form_diff

df_prepared["Recent_Surface_Form_Diff"] = recent_surface_form_diff

df_prepared["Fatigue_Diff"] = fatigue_diff

# Elo values are already well behaved; clipping is unnecessary for trees and
# calculating limits on the complete dataset would leak test-period information.
df_prepared["Elo_Diff"] = elo_diff
df_prepared["Surface_Elo_Diff"] = surface_elo_diff

# --- H2H ---
df_prepared["H2H_Diff"] = h2h_diff
df_prepared["H2H_Surface_Diff"] = h2h_surface_diff


numerical_features += [
    "Recent_Form_Diff", "Recent_Surface_Form_Diff", "Dominance_Form_Diff",
    
    "Fatigue_Diff",

    "Elo_Diff", "Surface_Elo_Diff",

    "H2H_Diff", "H2H_Surface_Diff",
]

In [310]:
"""Print computed ELO ratings"""

# sort elo ratings and print
sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
# filter "Sinner J." surface ratigs
# for surface in ["Hard", "Clay", "Grass"]:
    # print(f"Sinner J. {surface} Elo: {surface_elo_ratings.get(('Sinner J.', surface), STARTING_ELO)}")

[('Sinner J.', 2076.474175499717),
 ('Alcaraz C.', 2030.1235382488574),
 ('Djokovic N.', 2015.031088010582),
 ('Federer R.', 2001.6288065561894),
 ('Nadal R.', 1975.7785036086416),
 ('Zverev A.', 1911.5176542912436),
 ('Del Potro J.M.', 1876.5676313734211),
 ('Medvedev D.', 1844.12819913942),
 ('Soderling R.', 1840.903689409668),
 ('Fritz T.', 1822.9677735084365),
 ('Draper J.', 1822.291289187726),
 ('De Minaur A.', 1807.4172958099462),
 ('Fils A.', 1802.343228091823),
 ('Roddick A.', 1794.1222784891256),
 ('Auger Aliassime F.', 1781.6512706874962),
 ('Rune H.', 1779.606835919134),
 ('Paul T.', 1778.510271837399),
 ('Ruud C.', 1776.9632090500545),
 ('Raonic M.', 1770.157721324902),
 ('Musetti L.', 1769.7751748653516),
 ('Shelton B.', 1769.322169191497),
 ('Berdych T.', 1768.6459279738308),
 ('Jodar R.', 1765.5385218882018),
 ('Kyrgios N.', 1765.2123243117292),
 ('Bautista R.', 1757.650911148881),
 ('Lehecka J.', 1753.5682236700022),
 ('Rublev A.', 1752.5806462659248),
 ('Dimitrov G.', 

### Home factor

A player performs better when playing in his home country.

In [ ]:
# All cities
print(sorted(df_raw["Location"].unique()))

# some cities have end spaces (e.g. "Estoril ")
df_prepared["Location"] = df_prepared["Location"].str.strip()

LOCATION_TO_IOC = {
    "'s-Hertogenbosch": "NED", "Amersfoort": "NED", "Rotterdam": "NED",
    "Acapulco": "MEX", "Los Cabos": "MEX",
    "Adelaide": "AUS", "Brisbane": "AUS", "Melbourne": "AUS", "Sydney": "AUS",
    "Almaty": "KAZ", "Nur-Sultan": "KAZ",
    "Antalya": "TUR", "Istanbul": "TUR",
    "Antwerp": "BEL", "Brussels": "BEL",
    "Athens": "GRE",
    "Atlanta": "USA", "Cincinnati": "USA", "Dallas": "USA", "Delray Beach": "USA",
    "Houston": "USA", "Indian Wells": "USA", "Indianapolis": "USA", "Las Vegas": "USA",
    "Los Angeles": "USA", "Memphis": "USA", "Miami": "USA", "New Haven": "USA",
    "New York": "USA", "Newport": "USA", "San Diego": "USA", "San Jose": "USA",
    "Washington": "USA", "Winston-Salem": "USA",
    "Auckland": "NZL",
    "Bangkok": "THA",
    "Banja Luka": "BIH",
    "Barcelona": "ESP", "Gijon": "ESP", "Madrid": "ESP", "Mallorca": "ESP",
    "Marbella": "ESP", "Valencia": "ESP",
    "Basel": "SUI", "Geneva": "SUI", "Gstaad": "SUI",
    "Bastad": "SWE", "Stockholm": "SWE",
    "Beijing": "CHN", "Chengdu": "CHN", "Hangzhou": "CHN", "Shanghai": "CHN",
    "Shenzhen": "CHN", "Zhuhai": "CHN",
    "Belgrade": "SRB",
    "Bogota": "COL",
    "Bucharest": "ROU",
    "Budapest": "HUN",
    "Buenos Aires": "ARG", "Cordoba": "ARG",
    "Cagliari": "ITA", "Florence": "ITA", "Napoli": "ITA", "Palermo": "ITA",
    "Parma": "ITA", "Rome": "ITA", "Sardinia": "ITA", "Turin": "ITA",
    "Casablanca": "MAR", "Marrakech": "MAR",
    "Chennai": "IND", "Mumbai": "IND", "Pune": "IND",
    "Cologne": "GER", "Dusseldorf": "GER", "Halle": "GER", "Hamburg": "GER",
    "Munich": "GER", "Stuttgart": "GER",
    "Costa Do Sauipe": "BRA", "Rio de Janeiro": "BRA", "Sao Paulo": "BRA",
    "Doha": "QAT",
    "Dubai": "UAE",
    "Eastbourne": "GBR", "London": "GBR", "Nottingham": "GBR", "Queens Club": "GBR",
    "Estoril": "POR", "Oeiras": "POR",
    "Ho Chi Min City": "VIE",
    "Hong Kong": "HKG",
    "Johannesburg": "RSA",
    "Kitzbuhel": "AUT", "Portschach": "AUT", "Vienna": "AUT",
    "Kuala Lumpur": "MAS",
    "Lyon": "FRA", "Marseille": "FRA", "Metz": "FRA", "Montpellier": "FRA",
    "Nice": "FRA", "Paris": "FRA",
    "Monte Carlo": "MON",
    "Montreal": "CAN", "Toronto": "CAN",
    "Moscow": "RUS", "St. Petersburg": "RUS",
    "Quito": "ECU",
    "Santiago": "CHI", "Vina del Mar": "CHI",
    "Seoul": "KOR",
    "Singapore": "SIN",
    "Sofia": "BUL",
    "Sopot": "POL", "Warsaw": "POL",
    "Tel Aviv": "ISR",
    "Tokyo": "JPN",
    "Umag": "CRO", "Zagreb": "CRO",
}

# Unmapped cities. Should be an empty set
unmapped = set(df_prepared["Location"].unique()) - set(LOCATION_TO_IOC)
print("Città non mappate:", unmapped)

# Map the location to IOC (e.g. "Rome" -> "ITA") 
df_prepared["Location_IOC"] = df_prepared["Location"].map(LOCATION_TO_IOC)

# Get player IOC 
player_ioc = players_df.dropna(subset=["player_key", "ioc"]).set_index("player_key")["ioc"].to_dict()

df_prepared["Home_Player_1"] = (df_prepared["Player_1"].map(player_ioc) == df_prepared["Location_IOC"])
df_prepared["Home_Player_2"] = (df_prepared["Player_2"].map(player_ioc) == df_prepared["Location_IOC"])

# Compute the difference in home advantage between Player 1 and Player 2.
df_prepared["Home_Diff"] = df_prepared["Home_Player_1"].astype(int) - df_prepared["Home_Player_2"].astype(int)

numerical_features += ["Home_Diff"]

print(df_prepared["Home_Diff"].value_counts())


["'s-Hertogenbosch", 'Acapulco', 'Adelaide', 'Almaty', 'Amersfoort', 'Antalya', 'Antwerp', 'Athens', 'Atlanta', 'Auckland', 'Bangkok', 'Banja Luka', 'Barcelona', 'Basel', 'Bastad', 'Beijing', 'Belgrade', 'Bogota', 'Brisbane', 'Brussels', 'Bucharest', 'Budapest', 'Buenos Aires', 'Cagliari', 'Casablanca', 'Chengdu', 'Chennai', 'Cincinnati', 'Cologne', 'Cordoba', 'Costa Do Sauipe', 'Dallas', 'Delray Beach', 'Doha', 'Dubai', 'Dubai ', 'Dusseldorf', 'Eastbourne', 'Estoril', 'Estoril ', 'Florence', 'Geneva', 'Gijon', 'Gstaad', 'Halle', 'Hamburg', 'Hangzhou', 'Ho Chi Min City', 'Hong Kong', 'Houston', 'Indian Wells', 'Indianapolis', 'Istanbul', 'Johannesburg ', 'Kitzbuhel', 'Kuala Lumpur', 'Las Vegas', 'London', 'Los Angeles', 'Los Cabos', 'Lyon', 'Madrid', 'Mallorca', 'Marbella', 'Marrakech', 'Marseille', 'Melbourne', 'Memphis', 'Metz', 'Miami', 'Monte Carlo', 'Montpellier', 'Montreal', 'Moscow', 'Mumbai', 'Munich', 'Napoli', 'New Haven', 'New York', 'Newport', 'Nice', 'Nottingham', 'Nur-Sul

#### Categorical features

We create an imputer to manage missing values, using a **most frequent** strategy and **One hot encoding** to convert categorical features into numerical ones.

In [312]:
# Context columns are kept for debugging and error analysis, but the ablation
# tests showed no stable gain when they were used by the models.
categorical_features = []

We manage high cardinality categorical features, such as **Tournament**, by keeping only the most frequent values and grouping the others into a single category called **Other**.

In [313]:
high_cardinality_categorical_features = []


#### Pipeline

Finally, we can create an **imputer** to handle missing values in the dataset. We will use this later, after the splitting phase, to avoid data leakage.

In [314]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Create a preprocessor that combines both numeric and categorical pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
])

We will use it later during the model training.

#### Splitting and save the dataset

Now we can split the dataset into training and test sets

In [315]:
debug_features = [
    "Date", "Player_1", "Player_2", "Series", "Surface", "Round",
    "Odds_1", "Odds_2" # for baseline computation
]
features = debug_features + numerical_features + ["y"]

df_prepared = df_prepared[features].copy()

train_df = df_prepared.iloc[:raw_split_index].copy()
test_df = df_prepared.iloc[raw_split_index:].copy()


And save them as new `xlsx` datasets, to be used in the next notebook for model training and evaluation.

In [316]:
from pathlib import Path

TRAINING_PATH = Path("../data/prepared/tennis_training.xlsx")
TESTING_PATH = Path("../data/prepared/tennis_testing.xlsx")

TRAINING_PATH.parent.mkdir(parents=True, exist_ok=True)
TESTING_PATH.parent.mkdir(parents=True, exist_ok=True)

train_df.to_excel(TRAINING_PATH, index=False)
test_df.to_excel(TESTING_PATH, index=False)